# 1. Prep data & checkpoints

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p ckpt/
!wget https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth -O ckpt/jx_vit_base_p16_224-80ecf9dd.pth

In [ ]:
# get benchmark datasets (PAR_public_benchmark_datasets)
# Expected layout after unzipping (relative to /content/):
#   PA100k/dataset.pkl  +  PA100k/data/
#   MSP60k/dataset_random.pkl  +  MSP60k/images/
#   EventPAR/annotation_EventPAR/dataset_reorder.pkl  +  EventPAR/<sequences>/
#   DUKE/pad_duke.pkl  +  pad_duke_dataset_event/<tracks>/
!unzip -q drive/MyDrive/yomikawa-reid/datasets/OpenPAR.zip -d ./

# 2. Get UniPAR source code

In [ ]:
!unzip -q drive/MyDrive/yomikawa-reid/src/UniPAR.zip -d ./
!mv UniPAR/* .
!pip install -r requirements.txt

# 3. Train

Trains UniPAR with the multi-dataset setting from the paper (MSP60k + DUKE + EventPAR).

Key args:
- `--PE_load`: load ViT patch-embedding weights from the pretrained checkpoint
- `--dataset`: which datasets to use (space-separated)
- `--data_root`: root directory where datasets are stored (defaults to `.`)
- `--pretrain_path`: path to the ViT-Base pretrained checkpoint
- `--epoch 200`: number of training epochs
- `--epoch_save_ckpt 20`: save a checkpoint every N epochs

In [ ]:
!python mix_train.py \
    --PE_load \
    --dataset MSP60k DUKE EventPAR \
    --data_root . \
    --pretrain_path ckpt/jx_vit_base_p16_224-80ecf9dd.pth \
    --gpus 0 \
    --batchsize 8 \
    --epoch 200 \
    --epoch_save_ckpt 20 \
    --save_place multiDataset

# 4. Save checkpoint to Drive

In [ ]:
import glob, shutil, os

save_dir = '/content/drive/MyDrive/yomikawa-reid/ckpts/UniPAR'
os.makedirs(save_dir, exist_ok=True)

ckpts = sorted(glob.glob('logs/multiDataset/**/*.pth', recursive=True))
for ckpt in ckpts:
    dest = os.path.join(save_dir, os.path.basename(ckpt))
    shutil.copy2(ckpt, dest)
    print(f'Saved: {dest}')